In [ ]:
"""
Inference for Text3DSAM
Change Text Prompt in Cell 3
"""
# ============================================================================
# SECTION 1: SETUP & INSTALLATION
# ============================================================================
#Install Dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate huggingface_hub
!pip install -q nibabel SimpleITK pandas scikit-learn scipy scikit-image
!pip install -q monai einops timm sentencepiece protobuf
!pip install -q safetensors calflops


!pip install -q opencv-python Pillow matplotlib

print("Dependencies installed")

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')


import os
# Install kaggle
!pip install -q kaggle
# Create kaggle directory
!mkdir -p ~/.kaggle

# Check if kaggle.json already exists
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("\n Upload kaggle.json")
    from google.colab import files
    uploaded = files.upload()

    # Move kaggle.json to correct location
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download dataset
DATASET_PATH = "/content/dataset/lgg-mri-segmentation/kaggle_3m"
if not os.path.exists(DATASET_PATH):
    !kaggle datasets download -d mateuszbuda/lgg-mri-segmentation
    !unzip -q lgg-mri-segmentation.zip -d /content/dataset

    # Verify extraction
    if os.path.exists(DATASET_PATH):
        print("Dataset extracted")
    else:
        # Check alternate
        import glob
        data_csv_files = glob.glob("/content/dataset/**/data.csv", recursive=True)
        if data_csv_files:
            DATASET_PATH = os.path.dirname(data_csv_files[0])
else:
    print("Dataset already exists")

print(f"\nDataset location: {DATASET_PATH}")

# Clone Text3DSAM repo
import sys
if not os.path.exists('/content/Text3DSAM'):
    !git clone https://github.com/mirthAI/Text3DSAM.git
print("Repository cloned")

# Setup Python path
os.chdir('/content/Text3DSAM')
sys.path.insert(0, '/content/Text3DSAM')
sys.path.insert(0, '/content/Text3DSAM/src')



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 27.5 MB/s eta 0:00:00
Dependencies installed
Mounted at /content/drive

 Upload kaggle.json


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation
License(s): CC-BY-NC-SA-4.0
 99% 706M/714M [00:11<00:00, 164MB/s]
100% 714M/714M [00:11<00:00, 66.7MB/s]
Dataset extracted

Dataset location: /content/dataset/lgg-mri-segmentation/kaggle_3m
Cloning into 'Text3DSAM'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 30 (delta 2), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 27.02 KiB | 4.50 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Repository cloned


In [ ]:
#Download Text3DSAM

from huggingface_hub import snapshot_download


MODEL_DIR = "/content/models/Text3DSAM"

if not os.path.exists(f"{MODEL_DIR}/model.safetensors"):
    snapshot_download(
        repo_id="MagicXin/Text3DSAM",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False
    )
    print(f"Model downloaded to: {MODEL_DIR}")
else:
    print(f"Model already exists at: {MODEL_DIR}")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/119M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

...

training_args.bin:   0%|          | 0.00/7.25k [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

Model downloaded to: /content/models/Text3DSAM


In [ ]:
# Paths
OUTPUT_PATH = "/content/drive/MyDrive/SAM_results_text3dsam2"
os.makedirs(OUTPUT_PATH, exist_ok=True)

#ENTER TEXT PROMPT HERE ***********
TEXT_PROMPT = "Lower-grade giloma in brain MRI"

print(f"  Dataset: {DATASET_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Prompt: {TEXT_PROMPT}")

  Dataset: /content/dataset/lgg-mri-segmentation/kaggle_3m
  Output: /content/drive/MyDrive/SAM_results_text3dsam2
  Prompt: Lower-grade giloma in brain MRI


In [ ]:
#Functions
import torch
import numpy as np
import nibabel as nib
import pandas as pd
from PIL import Image
from glob import glob
import subprocess
from sklearn.metrics import f1_score, jaccard_score

def load_patient_volume_from_tif(patient_dir):
    """
    Load all TIF slices into a 3D volume

    Returns:
        volume: numpy array (D, H, W)
        image_files: list of image file paths
    """
    # Get all image files
    image_files = sorted([
        f for f in glob(os.path.join(patient_dir, "*.tif"))
        if f[-5].isdigit() and "_mask" not in f
    ])

    if len(image_files) == 0:
        return None, None

    # Load all slices
    slices = []
    for img_file in image_files:
        img = np.array(Image.open(img_file).convert('L'))
        slices.append(img)

    # Stack into 3D volume (D, H, W)
    volume = np.stack(slices, axis=0).astype(np.float32)

    return volume, image_files

def run_text3dsam_inference(volume, text_prompt):
    """
    Run Text3DSAM inference on a volume using the official pred.py script

    Args:
        volume: numpy array of shape (D, H, W)
        text_prompt: text description of what to segment

    Returns:
        prediction mask of same shape as input
    """
    # Create temp directories
    temp_input = "/content/temp_input"
    temp_output = "/content/temp_output"
    os.makedirs(temp_input, exist_ok=True)
    os.makedirs(temp_output, exist_ok=True)

    try:
        # Save volume as NPZ in the format pred.py expects
        npz_path = os.path.join(temp_input, "scan.npz")
        text_prompts = {
            "1": text_prompt,
            "instance_label": 1
        }
        np.savez_compressed(npz_path, imgs=volume, text_prompts=text_prompts)

        # Build command
        command = [
            "python", "src/pred.py",
            "--model_path", MODEL_DIR,
            "--input_dir", temp_input,
            "--output_dir", temp_output
        ]

        # Set up environment with proper PYTHONPATH
        env = os.environ.copy()
        env['PYTHONPATH'] = '/content/Text3DSAM:/content/Text3DSAM/src:' + env.get('PYTHONPATH', '')

        # Run inference
        result = subprocess.run(
            command,
            cwd='/content/Text3DSAM',
            env=env,
            capture_output=True,
            text=True,
            timeout=300
        )

        if result.returncode != 0:
            raise RuntimeError(f"pred.py failed: {result.stderr[:500]}")

        # Load prediction
        output_files = glob(f"{temp_output}/*.npz")
        if not output_files:
            raise RuntimeError("No output generated")

        result_data = np.load(output_files[0], allow_pickle=True)

        # Handle different possible key names
        if 'segs' in result_data:
            prediction = result_data['segs']
        elif 'prediction' in result_data:
            prediction = result_data['prediction']
        elif 'mask' in result_data:
            prediction = result_data['mask']
        else:
            prediction = result_data[list(result_data.keys())[0]]

        # Convert to binary (0 or 1)
        prediction = (prediction > 0).astype(np.uint8)

        return prediction

    finally:
        # Cleanup temp directories
        import shutil
        if os.path.exists(temp_input):
            shutil.rmtree(temp_input)
        if os.path.exists(temp_output):
            shutil.rmtree(temp_output)

def process_patient(patient_dir, patient_id, text_prompt):
    """
    Process a single patient
    """
    volume, image_files = load_patient_volume_from_tif(patient_dir)

    if volume is None:
        print(f"No images found")
        return None

    print(f"    Volume shape: {volume.shape}")

    # Segment using Text3DSAM
    pred_mask = run_text3dsam_inference(volume, text_prompt)

    print(f"    Segmentation shape: {pred_mask.shape}")

    return pred_mask, image_files

In [ ]:
#Process Patients + Perform Inference

# Load patient info
data_csv = os.path.join(DATASET_PATH, "data.csv")
patient_info = pd.read_csv(data_csv)
patient_dirs = sorted(glob(os.path.join(DATASET_PATH, "TCGA_*")))

print(f"Patients in dataset: {len(patient_info)}")
print(f"Patient directories found: {len(patient_dirs)}")

results = []

for idx, patient_dir in enumerate(patient_dirs, 1):
    full_patient_id = os.path.basename(patient_dir)
    short_patient_id = '_'.join(full_patient_id.split('_')[:3])

    print(f"\n[{idx}/{len(patient_dirs)}] {short_patient_id}")

    # Get demographics
    patient_row = patient_info[patient_info['Patient'] == short_patient_id]
    if len(patient_row) == 0:
        print("  No demographics, skipping")
        continue

    race = patient_row.iloc[0]['race']
    gender = patient_row.iloc[0]['gender']
    print(f"  Race: {race}, Gender: {gender}")

    try:
        # Process patient
        result = process_patient(patient_dir, short_patient_id, TEXT_PROMPT)

        if result is None:
            continue

        pred_volume, image_files = result

        print(f" Segmentation complete")

        # Save NIfTI
        affine = np.eye(4)
        out_path = os.path.join(OUTPUT_PATH, f"{short_patient_id}_text3dsam_predictions.nii.gz")
        nib.save(nib.Nifti1Image(pred_volume, affine), out_path)
        print(f"  Saved: {out_path}")

        # Calculate metrics against ground truth
        gt_pairs = []
        for img_file in image_files:
            base = os.path.splitext(img_file)[0]
            mask_file = f"{base}_mask.tif"
            if os.path.exists(mask_file):
                gt_pairs.append((img_file, mask_file))

        ious = []
        f1s = []

        for z, (_, mask_file) in enumerate(gt_pairs):
            if z >= pred_volume.shape[0]:
                break

            # Load GT
            gt_slice = np.array(Image.open(mask_file).convert("L"))
            gt_binary = (gt_slice > 0).astype(np.uint8)

            # Get prediction slice
            pred_slice = pred_volume[z]

            # Resize if needed
            if gt_binary.shape != pred_slice.shape:
                from scipy.ndimage import zoom
                scale_factors = [gt_binary.shape[0] / pred_slice.shape[0],
                               gt_binary.shape[1] / pred_slice.shape[1]]
                pred_slice = zoom(pred_slice, scale_factors, order=0).astype(np.uint8)

            # Calculate metrics
            gt_sum = gt_binary.sum()
            pred_sum = pred_slice.sum()

            # Case 1: both empty → perfect score
            if gt_sum == 0 and pred_sum == 0:
                ious.append(1.0)
                f1s.append(1.0)

            # Case 2: one empty, one not → zero score
            elif gt_sum == 0 and pred_sum > 0:
                ious.append(0.0)
                f1s.append(0.0)

            elif gt_sum > 0 and pred_sum == 0:
                ious.append(0.0)
                f1s.append(0.0)

            # Case 3: both non-empty → compute normally
            else:
                ious.append(
                    jaccard_score(
                        gt_binary.flatten(),
                        pred_slice.flatten(),
                        zero_division=0
                    )
                )
                f1s.append(
                    f1_score(
                        gt_binary.flatten(),
                        pred_slice.flatten(),
                        zero_division=0
                    )
                )

        mean_iou = np.mean(ious) if ious else 0
        mean_f1 = np.mean(f1s) if f1s else 0

        print(f"  IoU: {mean_iou:.4f}, F1: {mean_f1:.4f}")

        results.append({
            'Patient': short_patient_id,
            'Race': race,
            'Gender': gender,
            'IoU': mean_iou,
            'F1': mean_f1,
            'Num_Slices': len(ious),
            'Prediction_Path': out_path
        })

    except Exception as e:
        print(f"  ❌ Error: {str(e)[:200]}")
        import traceback
        traceback.print_exc()
        continue

Patients in dataset: 110
Patient directories found: 110

[1/110] TCGA_CS_4941
  Race: 3.0, Gender: 2.0
    Volume shape: (23, 256, 256)
    Segmentation shape: (23, 256, 256)
 Segmentation complete
  Saved: /content/drive/MyDrive/SAM_results_text3dsam2/TCGA_CS_4941_text3dsam_predictions.nii.gz
  IoU: 0.5016, F1: 0.5430

[2/110] TCGA_CS_4942
  Race: 2.0, Gender: 1.0
    Volume shape: (20, 256, 256)
    Segmentation shape: (20, 256, 256)
 Segmentation complete
  Saved: /content/drive/MyDrive/SAM_results_text3dsam2/TCGA_CS_4942_text3dsam_predictions.nii.gz
  IoU: 0.1034, F1: 0.1318

[3/110] TCGA_CS_4943
  Race: 3.0, Gender: 2.0
    Volume shape: (20, 256, 256)
    Segmentation shape: (20, 256, 256)
 Segmentation complete
  Saved: /content/drive/MyDrive/SAM_results_text3dsam2/TCGA_CS_4943_text3dsam_predictions.nii.gz
  IoU: 0.1705, F1: 0.2316

[4/110] TCGA_CS_4944
  Race: 3.0, Gender: 2.0
    Volume shape: (20, 256, 256)
    Segmentation shape: (20, 256, 256)
 Segmentation complete
  Saved

In [ ]:
#Process Results
if len(results) > 0:
    results_df = pd.DataFrame(results)

    # Save detailed results
    csv_path = os.path.join(OUTPUT_PATH, "text3dsam_patient_metrics.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path}")

    # Overall metrics
    overall_iou = results_df['IoU'].mean()
    overall_f1 = results_df['F1'].mean()

    print(f"  Mean IoU: {overall_iou:.4f}")
    print(f"  Mean F1:  {overall_f1:.4f}")
    print(f"  Patients: {len(results_df)}")

    # Race-level metrics
    race_stats = results_df.groupby('Race')[['IoU', 'F1']].agg(['mean', 'std', 'count']).round(4)
    print(f"\n{'='*70}")
    print("RACE LEVEL METRICS")
    print(f"{'='*70}")
    print(race_stats)
    race_stats.to_csv(os.path.join(OUTPUT_PATH, "text3dsam_race_metrics.csv"))

    # Gender-level metrics
    gender_stats = results_df.groupby('Gender')[['IoU', 'F1']].agg(['mean', 'std', 'count']).round(4)
    print(f"\n{'='*70}")
    print("GENDER LEVEL METRICS")
    print(f"{'='*70}")
    print(gender_stats)
    gender_stats.to_csv(os.path.join(OUTPUT_PATH, "text3dsam_gender_metrics.csv"))

    # Save summary
    summary_path = os.path.join(OUTPUT_PATH, "text3dsam_summary.txt")
    with open(summary_path, 'w') as f:
        f.write("OVERALL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(f"Mean IoU: {overall_iou:.4f}\n")
        f.write(f"Mean F1:  {overall_f1:.4f}\n")
        f.write(f"Patients: {len(results_df)}\n\n")

        f.write("RACE-LEVEL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(race_stats.to_string())
        f.write("\n\n")

        f.write("GENDER-LEVEL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(gender_stats.to_string())
        f.write("\n")

    print(f" Saved: {summary_path}")

    print(f"\nResults saved to: {OUTPUT_PATH}")
else:
    print(" No results to save")

  Saved: /content/drive/MyDrive/SAM_results_text3dsam2/text3dsam_patient_metrics.csv
  Mean IoU: 0.3672
  Mean F1:  0.3924
  Patients: 110

RACE LEVEL METRICS
         IoU                    F1              
        mean     std count    mean     std count
Race                                            
2.0   0.3504  0.1598    10  0.3728  0.1733    10
3.0   0.3693  0.1516    98  0.3953  0.1621    98

GENDER LEVEL METRICS
           IoU                    F1              
          mean     std count    mean     std count
Gender                                            
1.0     0.3713  0.1712    56  0.3957  0.1840    56
2.0     0.3662  0.1292    53  0.3928  0.1367    53
 Saved: /content/drive/MyDrive/SAM_results_text3dsam2/text3dsam_summary.txt

Results saved to: /content/drive/MyDrive/SAM_results_text3dsam2


In [ ]:
# ---------------- Excel-friendly summary ----------------

# Sort keys to keep order stable
race_means = race_stats['IoU']['mean'].sort_index()
gender_means = gender_stats['IoU']['mean'].sort_index()

# Build header
header_cols = ["IoU"]
header_cols += [f"Race {race} IoU" for race in race_means.index]
header_cols += [f"Gender {gender} IoU" for gender in gender_means.index]

# Build values row
value_cols = [overall_iou]
value_cols += list(race_means.values)
value_cols += list(gender_means.values)

# Print for Excel copy-paste
print("\n" + "="*70)
print("EXCEL-FRIENDLY SUMMARY (TAB SEPARATED)")
print("="*70)

print("\t".join(header_cols))
print("\t".join(f"{v:.6f}" for v in value_cols))



EXCEL-FRIENDLY SUMMARY (TAB SEPARATED)
IoU	Race 2.0 IoU	Race 3.0 IoU	Gender 1.0 IoU	Gender 2.0 IoU
0.367184	0.350400	0.369300	0.371300	0.366200
